# Implementing Multiple Turns

### Stop Reason
Tells why Claude stop generating text
`tool_use`:
- Claude decided it needs to call a tool
`end_turn`:
- Claude finished generating it's assistant message
`max_tokens`: 
- Claude has hit the token output limit
`stop_sequences`:
- It has encountered a stop sequence in the output

In [1]:
from utils.util_funcs import *
import json 

client, model = api_client_setup()

In [2]:
def run_tool(tool_name, tool_input):
    if tool_name == 'get_current_datetime':
        return get_current_datetime(**tool_input)

def run_tools(message): 
    tool_requests = [
        block for block in message.content if block.type == 'tool_use'
    ]

    tool_result_blocks = []

    for tool_request in tool_requests:
        if tool_request.name == 'get_current_datetime':
            try: 
                tool_output = run_tool(tool_request.name, tool_request.input)
                tool_result_block = { 
                    "type": "tool_result",
                    "tool_use_id": tool_request.id,
                    "content": json.dumps(tool_output),
                    "is_error": False
                }
            except Exception as e: 
                tool_result_block = { 
                    "type": "tool_result",
                    "tool_use_id": tool_request.id,
                    "content": f"Error: {e}",
                    "is_error": True
                }

            tool_result_blocks.append(tool_result_block)

    return tool_result_blocks

In [3]:
def run_conversation(messages):
    while True: 
        response = chat(messages, tools = [get_current_datetime_schema])

        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break
        
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

In [4]:
messages = []
add_user_message(
    messages, 
    "what is the current time in HH:MM format? Also, what is the current time in SS format?"
)
# messages
run_conversation(messages)


The current time is:
- **HH:MM format**: 11:20
- **SS format**: 19 (seconds)

So the full current time is 11:20:19.
